# Score the detectors on DiFF and DiffusionFace

Scores the lab's photo detectors (v1, v2), the official GenD checkpoints and their fusions on the DiFF test split built by `build_diff_test.ipynb`, the `cross_diff` protocol, or on the DiffusionFace sample of `build_diffusionface_test.ipynb` (`cross_diffusionface`), chosen by `TEST_SET`. None of them trained on either. DiFF's paper and the methods compared on it report one AUC per subset (T2I, I2I, FS, FE): every pristine image against that subset's fakes, which `auc_by_group` computes.

Until the pristine images arrive the dataset holds only fakes. The notebook then reports, per subset, the mean attack score and the share of fakes scored above 0.5; no AUC exists without the bona fide side.

**Fair comparison:** published DiFF numbers come from detectors trained on FaceForensics++ alone. v1 and v2 trained on DeepFakeFace, which holds Stable Diffusion faces, so their DiFF numbers are not comparable to those without saying so.

**Runtime:** Colab Pro with a GPU. **Needs** the Colab secret `HF_TOKEN`.

### Configuration

In [ ]:
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
TEST_SET = "cross_diff"  # or "cross_diffusionface"
TEST_DATASETS = {"cross_diff": "paulohenriquevn/face-decision-diff-test",
                 "cross_diffusionface": "paulohenriquevn/face-decision-diffusionface-test"}
HF_MODEL = "paulohenriquevn/face-decision-clip-ln"  # reports go to HF_MODEL/pretrained/<detector>/<TEST_SET>.json
CLIP_MODEL = "openai/clip-vit-large-patch14"
GEND_MODELS = {"clip": "yermandy/GenD_CLIP_L_14", "pe": "yermandy/GenD_PE_L"}
GEND_FACE_SCALE = 1.3
# Photo experts: v1 (subject_holdout checkpoint of commit b66500a) and v2 (lab_ffpp_real, commit b962286).
V1_CHECKPOINT = "subject_holdout/trained_parameters.pt"
V2_CHECKPOINT = "openai-ce-e1-b32-lr0.0003-fs1.5-g10-noaug/lab_ffpp_real/checkpoint.pt"

In [ ]:
%%capture
!pip install -q "face-decision-lab @ {REPO_SOURCE}" "timm>=1.0.20"

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token = userdata.get("HF_TOKEN"))

### Load the test set

In [ ]:
from face_decision_lab.hub import load_hub_samples

samples, hub_images = load_hub_samples(TEST_DATASETS[TEST_SET], key_prefix = TEST_SET)
images = dict(zip([s.image_path for s in samples], hub_images, strict = True))
# Both protocols test on every crop of their dataset; nothing here trains, so the lab data is not loaded.
test = samples
bona_fide = sum(s.is_bona_fide for s in test)
print(len(test), "crops,", bona_fide, "bona fide")

### Detectors

In [ ]:
import importlib.util, io, json
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import requests
import torch
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image
from safetensors.torch import load_file
from torch import nn
from tqdm.auto import tqdm
from transformers import CLIPImageProcessor, CLIPVisionModel, CLIPVisionModelWithProjection
from face_decision_lab.faces import tighten_crop
from face_decision_lab.rows import open_image

api = HfApi()
pool = ThreadPoolExecutor(max_workers = 8)
EXAMPLES = ["https://github.com/yermandy/deepfake-detection/blob/main/datasets/FF/DF/000_003/000.png?raw=true",
            "https://github.com/yermandy/deepfake-detection/blob/main/datasets/FF/real/000/000.png?raw=true"]
example_frames = [Image.open(io.BytesIO(requests.get(u, timeout = 30).content)).convert("RGB") for u in EXAMPLES]

def load_gend(repo):
    spec = importlib.util.spec_from_file_location(f"modeling_{repo.split('/')[1]}", hf_hub_download(repo, "modeling_gend.py"))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    model = module.GenD(module.GenDConfig.from_pretrained(repo))
    loaded = model.load_state_dict(load_file(hf_hub_download(repo, "model.safetensors")), strict = False)
    assert not loaded.missing_keys and not loaded.unexpected_keys, loaded
    model = model.cuda().eval()
    with torch.no_grad():
        fake = model(torch.stack([model.feature_extractor.preprocess(f) for f in example_frames]).cuda()).softmax(-1)[:, 1].tolist()
    assert fake[0] > 0.5 > fake[1], (repo, fake)
    return model

processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL)

class LayerNormTunedCLIPv1(nn.Module):  # the v1 detector, as trained at commit b66500a
    def __init__(self):
        super().__init__()
        self.encoder = CLIPVisionModelWithProjection.from_pretrained(CLIP_MODEL)
        self.head = nn.Linear(self.encoder.config.projection_dim, 1)

    def forward(self, pixels):
        return torch.sigmoid(self.head(nn.functional.normalize(self.encoder(pixel_values = pixels).image_embeds, dim = -1)).squeeze(-1))

class LayerNormTunedCLIP(nn.Module):  # clip_layernorm_detector.ipynb's detector (pooled features, two-class head)
    def __init__(self):
        super().__init__()
        self.encoder = CLIPVisionModel.from_pretrained(CLIP_MODEL)
        self.head = nn.Linear(self.encoder.config.hidden_size, 2)

    def forward(self, pixels):
        return self.head(nn.functional.normalize(self.encoder(pixel_values = pixels).pooler_output, dim = -1)).softmax(-1)[:, 1]

def load_v1():
    model = LayerNormTunedCLIPv1()
    model.load_state_dict(torch.load(hf_hub_download(HF_MODEL, V1_CHECKPOINT)), strict = False)
    return model.cuda().eval()

def load_v2():
    model = LayerNormTunedCLIP()
    loaded = model.load_state_dict(torch.load(hf_hub_download(HF_MODEL, V2_CHECKPOINT), weights_only = False)["trainable"], strict = False)
    assert not loaded.unexpected_keys, loaded.unexpected_keys[:5]
    return model.cuda().eval()

def scores_of(part, prepare, model):
    out = []
    with torch.no_grad(), torch.autocast("cuda", dtype = torch.bfloat16):
        for i in tqdm(range(0, len(part), 128), leave = False):
            out += model(torch.stack(list(pool.map(prepare, part[i:i + 128]))).cuda()).float().cpu().tolist()
    return np.array(out)

def clip_pixels(sample):  # the lab detectors trained on the 1.5x crops as built
    return processor(images = open_image(images[sample.image_path]), return_tensors = "pt")["pixel_values"][0]

scores = {}
for name, repo in GEND_MODELS.items():
    model = load_gend(repo)

    def prepare(s, m = model):
        return m.feature_extractor.preprocess(tighten_crop(open_image(images[s.image_path]), GEND_FACE_SCALE))

    scores[f"gend_{name}"] = scores_of(test, prepare, lambda x, m = model: m(x).softmax(-1)[:, 1])
    del model
scores["gend_ensemble_clip_pe"] = (scores["gend_clip"] + scores["gend_pe"]) / 2
scores["layernorm_clip_v1"] = scores_of(test, clip_pixels, load_v1())
scores["v2_ffpp_real"] = scores_of(test, clip_pixels, load_v2())
scores["fusion_max_v1_gendens"] = np.maximum(scores["layernorm_clip_v1"], scores["gend_ensemble_clip_pe"])
scores["fusion_max_v2_ffpp_real_gendens"] = np.maximum(scores["v2_ffpp_real"], scores["gend_ensemble_clip_pe"])

### Reports

With bona fide crops: the full evaluation report plus the AUC of each DiFF subset (on DiffusionFace, `auc_by_generator` gives one AUC per folder). Without them: per subset, the mean attack score and the share of fakes above 0.5.

In [ ]:
from face_decision_lab.protocol import auc_by_group, evaluation_report
from face_decision_lab.sources import diff

KEYS = ("samples", "auc", "eer", "bpcer_at_max_apcer", "auc_by_generator")
# DiFF groups its methods into four subsets; DiffusionFace reports each folder on its own.
group = diff.subset if TEST_SET == "cross_diff" else (lambda generator: generator)
subsets = np.array([None if s.is_bona_fide else group(s.generator) for s in test])
names = sorted(set(subsets[subsets != None]))  # noqa: E711

def summary(values):
    if bona_fide:
        result = {k: v for k, v in evaluation_report(test, values.tolist()).items() if k in KEYS}
        result["auc_by_subset"] = auc_by_group(test, values.tolist(), group = group)
        result["mean_auc_over_subsets"] = float(np.mean(list(result["auc_by_subset"].values())))
    else:
        result = {"samples": len(test), "bona_fide": 0}
    result["mean_score_by_subset"] = {k: float(values[subsets == k].mean()) for k in names}
    result["share_above_half_by_subset"] = {k: float((values[subsets == k] > 0.5).mean()) for k in names}
    return result

for detector, values in scores.items():
    result = summary(values)
    api.upload_file(path_or_fileobj = json.dumps(result, indent = 2).encode(), repo_id = HF_MODEL,
                    path_in_repo = f"pretrained/{detector}/{TEST_SET}.json", commit_message = f"{detector} on {TEST_SET}")
    shown = "auc_by_subset" if bona_fide else "share_above_half_by_subset"
    print(f"{detector:34s}", {k: round(v, 4) for k, v in result[shown].items()})